# 02 - Seasonal Patterns

STL decomposition, ADF/KPSS stationarity tests, and ACF/PACF plots
for select repos to characterise periodic patterns in PR activity.

In [ ]:
from pathlib import Path

import pandas as pd

from oss_pulse.analyze.seasonal import (
    compute_acf_pacf,
    stl_decompose,
    test_stationarity,
)
from oss_pulse.visualize.timeseries import plot_acf_pacf, plot_decomposition
from oss_pulse.visualize.style import setup_style

setup_style()

In [ ]:
# Load weekly aggregated data
DATA_DIR = Path("../data/processed")
weekly_df = pd.read_parquet(DATA_DIR / "repo_weekly.parquet")

repos = weekly_df["repo_name"].unique()
print(f"Available repos: {len(repos)}")
print(repos[:5])

In [ ]:
# Pick the first repo for detailed analysis
# TODO: run with real data
target_repo = repos[0]
repo_weekly = (
    weekly_df[weekly_df["repo_name"] == target_repo]
    .sort_values("year_week")
    .reset_index(drop=True)
)

series = repo_weekly.set_index("year_week")["pr_count"]
print(f"Repo: {target_repo}")
print(f"Series length: {len(series)} weeks")

In [ ]:
# STL decomposition (period=52 for weekly data)
# TODO: run with real data
stl_result = stl_decompose(series, period=52)

fig = plot_decomposition(stl_result, title=f"STL Decomposition: {target_repo}")
fig.show()

In [ ]:
# ADF and KPSS stationarity tests
# TODO: run with real data
stationarity = test_stationarity(series)

print(f"Stationarity results for {target_repo}:")
for key, value in stationarity.items():
    print(f"  {key}: {value}")

# Run across all repos
stationarity_records = []
for repo in repos:
    s = (
        weekly_df[weekly_df["repo_name"] == repo]
        .sort_values("year_week")
        .set_index("year_week")["pr_count"]
    )
    if len(s) >= 30:
        result = test_stationarity(s)
        result["repo_name"] = repo
        stationarity_records.append(result)

stationarity_df = pd.DataFrame(stationarity_records)
print(f"\nStationary repos: {stationarity_df['is_stationary'].sum()} / {len(stationarity_df)}")
stationarity_df.head(10)

In [ ]:
# ACF / PACF plots
# TODO: run with real data
acf_vals, pacf_vals = compute_acf_pacf(series, nlags=min(52, len(series) // 2 - 1))

fig = plot_acf_pacf(acf_vals, pacf_vals, title=f"ACF / PACF: {target_repo}")
fig.show()